In [270]:
# packages
%pip install pandas
import pandas as pd
from mod02_build_bot_predictor import train_model

You should consider upgrading via the '/Users/josepomavilla/Programming/code_3000/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


### Define a function to extract predictions from the model

In [271]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [272]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [273]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [274]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [275]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [276]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [277]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 112,
 'tn': 2595,
 'fp': 42,
 'fn': 251,
 'misclassification_rate': 0.09766666666666667,
 'false_positive_rate': 0.015927189988623434,
 'false_negative_rate': 0.6914600550964187}

### Check results on the test set (new data not yet seen by the model)

In [278]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 33,
 'tn': 859,
 'fp': 15,
 'fn': 93,
 'misclassification_rate': 0.108,
 'false_positive_rate': 0.017162471395881007,
 'false_negative_rate': 0.7380952380952381}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

Although I was able to get the misclassification rate down to 10.8 percent compared to 12.6% which would be if we labeled every account a human, this does not suggest that we are accurate in predicting the bots. The model will still miss most bots even from the data it learned from and this can be seen by the false negative for training and testing we have about 69% and 74% approx, meaning that out of all of the bots we have, most of them will be missed. The margin only increased by a small portion, the learning model can see that a majority of accounts are human so 9/10 times its safer to call it a human vs bot account which is why there is that type of misclassfication rate, there is only so much i can do.

### What are potential ramifications of false positives from the model?

This is small, a majority of the time we are going to qualify an account as a human rather than a bot, the false positive rate is approx 2%. The person will be marked as a bot 2% of the times which is a small percentage compared to false negatives.

### What are potential ramifications of false negatives from the model?

It would cause a lot of bot accounts to go undetected, the model has data where a majority of these accounts are humans, so when it attempts to predict a bot, it is much safer prediciting it as a human which is why the false negative is much higher at about 74%.